In [ ]:
import re
import math
import os
import pandas as pd

# =============================================================================
# 1. Data Parser
# =============================================================================
def parse_dat_file(dat_filepath):
    # Defaults matching modified_model.mod and complex_solo_mod.dat
    params = {
        'scaleLength': 635.0,
        'stringSpacing': 10.8,
        'timePerMm': 0.25,
        's_max': 6, 'f_max': 22, 'p_max': 17,
        'fingerLengths_mm': [70.0, 75.0, 70.0, 50.0],
        'fingerDifficulty': [1.0, 1.2, 1.4, 1.5],
        'natural_min_spread_mm': [20.0, 40.0, 55.0],
        'availableTime': [],
        # Weights (matching complex_solo_mod.dat)
        'd_PC': 0.8, 'd_SC': 1.0, 'd_MC': 0.01, 'd_ST': 1.5, 'd_DH': 0.7, 'd_NO': 10.0,
        'alpha_PC': 0.2, 'alpha_SC': 0.2, 'alpha_MC': 0.2, 'alpha_ST': 0.2, 'alpha_DH': 0.2, 'alpha_NO': 0.2
    }

    print(f"DEBUG: Parsing .dat file: {dat_filepath}")
    if not os.path.exists(dat_filepath):
        print("WARNING: File not found. Using defaults.")
        return params

    try:
        content = ""
        for enc in ['utf-8-sig', 'utf-8', 'cp932']:
            try:
                with open(dat_filepath, 'r', encoding=enc) as f:
                    content = f.read()
                break
            except UnicodeDecodeError: continue

        def get_val(name, cast_func):
            pat = re.compile(rf'{name}\s*=\s*([-+]?[0-9]*\.?[0-9]+);?')
            m = pat.search(content)
            if m: return cast_func(m.group(1))
            return None

        def get_arr(name):
            pat = re.compile(rf'{name}\s*=\s*\[(.*?)\];?', re.DOTALL)
            m = pat.search(content)
            if m:
                txt = re.sub(r'//.*', '', m.group(1)).replace('\n', ' ')
                return [float(v) for v in txt.split(',') if v.strip()]
            return None

        for k in params.keys():
            if isinstance(params[k], list):
                val = get_arr(k)
                if val: params[k] = val
            else:
                val = get_val(k, float)
                if val is not None: params[k] = val
        
        params['s_max'] = int(params['s_max'])
        params['f_max'] = int(params['f_max'])
        params['p_max'] = int(params['p_max'])
        
        return params
    except Exception as e:
        print(f"ERROR: Parsing failed ({e}). Using defaults.")
        return params

def parse_solution_file(solution_filepath):
    notes_data = []
    if not os.path.exists(solution_filepath): return []
    
    try:
        lines = []
        for enc in ['utf-8-sig', 'utf-8', 'cp932']:
            try:
                with open(solution_filepath, 'r', encoding=enc) as f:
                    lines = f.readlines()
                break
            except UnicodeDecodeError: continue

        header_found = False
        for line in lines:
            line = line.strip()
            if not line: continue
            if not header_found:
                if "Note" in line and "String" in line: header_found = True
                continue
            if line.startswith('---'): break
            
            parts = re.split(r'\s+', line)
            if parts[0] == "Note": continue
            if len(parts) >= 5:
                try:
                    # Handle Rest Note 'R'
                    if parts[0] == 'R':
                         note = {
                            'note': 'R', 'string': 0, 'fret': 0, 'hand': 0, 'finger': 0
                        }
                    else:
                        note = {
                            'note': parts[0],
                            'string': int(parts[1]),
                            'fret': int(parts[2]),
                            'hand': int(parts[3]),
                            'finger': 0 if parts[4] == 'x' else int(parts[4])
                        }
                    notes_data.append(note)
                except ValueError: continue
        return notes_data
    except: return []

# =============================================================================
# 2. Model Logic (Synced with modified_model.mod v17)
# =============================================================================
class GuitarModelReproducer:
    def __init__(self, params):
        self.p = params
        
        self.fingerLengths = [0.0] + self.p['fingerLengths_mm']
        self.fingerDiff = self.p['fingerDifficulty']
        
        # Knuckle distances matching OPL: Index=0, Middle=20, Ring=40, Pinky=55
        self.knuckleDist = [0.0, 0.0, 20.0, 40.0, 55.0]
        
        # Limit Ratios matching OPL: Index=1.3, Middle/Ring=1.05, Pinky=1.1
        self.limitRatios = [0.0, 1.3, 1.05, 1.05, 1.1]
        
        # Base Factors matching OPL: Index=0.7, Middle=0.9, Ring=0.85, Pinky=1.0
        self.baseFactors = [0.0, 0.7, 0.9, 0.85, 1.0]
        
        self.stringTension = {1: 1.0, 2: 1.0, 3: 1.0, 4: 1.2, 5: 1.4, 6: 1.6}
        
        self.deg_to_rad = math.pi / 180.0
        self.rad_to_deg = 180.0 / math.pi
        self.breakAngleDeg = 15.0
        self.tanTheta = math.tan(self.breakAngleDeg * self.deg_to_rad)
        
        sl = self.p['scaleLength']
        self.fret_pos = [sl - (sl / (2 ** (f / 12.0))) for f in range(self.p['f_max'] + 1)]
        
        # 12F breakpoint for neck factor
        self.dist_12 = sl - (sl / (2 ** (12.0/12.0)))
        self.dist_max = sl - (sl / (2 ** (self.p['f_max']/12.0)))

        # Time Penalty Parameters (Power Law)
        self.tp_base = 0.15
        self.tp_ref = 60.0
        self.tp_exp = 1.5

    def calculate_costs(self, hand_pos, finger_idx, string, fret):
        """
        Returns (Stretch_Cost, Physics_Cost, Best_Wrist_Angle) matching OPL v17 logic
        """
        if fret < hand_pos: return (100000.0, 0.0, 0.0)

        pivotX = self.fret_pos[hand_pos]
        pivotY = 0.0

        # --- Physics Cost (angle-independent) ---
        a_mm = 0.5 + (2.0 * fret / 12.0)
        l_mm = self.p['scaleLength'] - self.fret_pos[fret]
        tau = self.stringTension.get(string, 1.0)
        
        if l_mm <= 0.001: forceIndex = 0.0
        else: forceIndex = (2.0 * tau * a_mm) / (l_mm * self.tanTheta)
        
        diff = self.fingerDiff[finger_idx - 1]
        ph_val = (forceIndex * 100.0) * diff

        # --- Stretch Cost with Angle Search (-90 to 0 degrees) ---
        minStCost = 100000.0
        bestAngle = 0.0
        foundValidAngle = False

        for ang in range(-90, 1, 2):  # -90 to 0, step 2
            thetaRad = ang * self.deg_to_rad
            
            # Knuckle position
            d = self.knuckleDist[finger_idx]
            kX = pivotX + (d * math.cos(thetaRad))
            kY = pivotY + (d * math.sin(thetaRad))

            # Target position
            tX = self.fret_pos[fret]
            tY = (string - 1) * self.p['stringSpacing']

            # Vector and distance
            vX = tX - kX
            vY = tY - kY
            dist = math.sqrt(vX*vX + vY*vY)

            # --- Neck Factor (Piecewise at 12F) ---
            dist_current = self.p['scaleLength'] - self.fret_pos[fret]
            neckFactor = 1.0
            
            reduction_at_break = 0.015  # 1.5% at 12F
            reduction_at_max = 0.15     # 15% at max fret

            if dist_current <= self.dist_12:
                # Phase 1: Nut -> 12F
                neckFactor = 1.0 - (reduction_at_break * (dist_current / self.dist_12))
            else:
                # Phase 2: 12F -> Max
                ratioHigh = 0.0
                if self.dist_max > self.dist_12:
                    ratioHigh = (dist_current - self.dist_12) / (self.dist_max - self.dist_12)
                if ratioHigh > 1.0: ratioHigh = 1.0
                neckFactor = 1.0 - (reduction_at_break + ((reduction_at_max - reduction_at_break) * ratioHigh))

            # Effective finger length
            effLen = self.fingerLengths[finger_idx] * self.baseFactors[finger_idx] * neckFactor

            # Distance hard limit check
            if effLen > 0 and dist > effLen * self.limitRatios[finger_idx]:
                continue

            # --- Angle Calculation ---
            nX = -math.sin(thetaRad)
            nY = math.cos(thetaRad)
            
            dotProd = vX * nX + vY * nY
            cosDelta = (dotProd / dist) if dist > 0.001 else 1.0
            cosDelta = max(-1.0, min(1.0, cosDelta))
            deltaAbsDeg = math.acos(cosDelta) * self.rad_to_deg

            # Direction check (cross product)
            crossProd = nX * vY - nY * vX
            isLowFretSide = 1 if crossProd > 0 else 0
            deltaSignedDeg = deltaAbsDeg if isLowFretSide == 1 else -deltaAbsDeg

            # --- Angle Hard Limits ---
            if isLowFretSide == 1:
                limitAngle = 15.0  # Inner limit
                relaxAdd = 5.0
            else:
                limitAngle = 30.0  # Outer limit
                relaxAdd = 10.0
            
            # Pinky relaxation (+5.0)
            if finger_idx == 4: limitAngle += 5.0

            # Close distance relaxation
            ratio = 0.0
            isCloseDist = False
            if effLen > 0:
                ratio = dist / effLen
                if ratio < 0.5:
                    limitAngle += relaxAdd
                    isCloseDist = True

            # Angle hard limit check
            if deltaAbsDeg > limitAngle:
                continue

            # --- Distance Cost ---
            distCost = 0.0
            distSlope = 10.0
            if finger_idx == 1: distSlope = 20.0  # Index steep
            if finger_idx == 3: distSlope = 25.0  # Ring steep

            if ratio < 0.3:
                distCost = (0.3 - ratio) * 5.0
            elif ratio <= 0.6:
                distCost = 0.0  # Sweet spot
            else:
                distCost = (ratio - 0.6) * distSlope
                if ratio > 1.0:
                    # Overstretch penalty (x30)
                    distCost += (ratio - 1.0) * 30.0

            # --- Angle Cost ---
            comfortAngle = 5.0
            angleCost = 0.0
            coeffPositive = 1.5  # Inner penalty
            coeffNegative = 1.0  # Outer penalty

            if deltaSignedDeg > comfortAngle:
                angleCost = (deltaSignedDeg - comfortAngle) * coeffPositive
            elif deltaSignedDeg < -comfortAngle:
                angleCost = (abs(deltaSignedDeg) - comfortAngle) * coeffNegative
            
            # Close distance halves angle cost
            if isCloseDist: angleCost *= 0.5

            # Total cost (without difficulty multiplication, matching OPL)
            currentCost = distCost + angleCost

            # Update minimum
            if currentCost < minStCost:
                minStCost = currentCost
                bestAngle = ang
                foundValidAngle = True

        if foundValidAngle:
            return (minStCost, ph_val, bestAngle)
        else:
            return (100000.0, ph_val, 0.0)

    def calculate_total(self, notes_data):
        total = {'PC': 0.0, 'SC': 0.0, 'MC': 0.0, 'ST': 0.0, 'DH': 0.0, 'NO': 0.0}
        details = []
        
        num_notes = len(notes_data)
        avail_time = self.p['availableTime']
        
        if len(avail_time) < num_notes:
            avail_time += [30.0] * (num_notes - len(avail_time))
        
        prev_idx_map = {}
        acc_time = {}

        # Time Calculation Logic
        for i in range(num_notes):
            is_rest = (notes_data[i]['note'] == 'R')
            if not is_rest:
                p = i - 1
                t_sum = 0.0
                while p >= 0 and notes_data[p]['note'] == 'R':
                    t_sum += avail_time[p]
                    p -= 1
                if p >= 0:
                    prev_idx_map[i] = p
                    acc_time[i] = t_sum + avail_time[p]
                else:
                    prev_idx_map[i] = i
                    acc_time[i] = 0.0
            else:
                acc_time[i] = 0.0
                prev_idx_map[i] = -1

        # Store wrist angles for MC calculation
        wrist_angles = [0.0] * num_notes

        # First pass: calculate wrist angles
        for i in range(num_notes):
            curr = notes_data[i]
            if curr['note'] != 'R' and curr['finger'] in [1,2,3,4] and curr['fret'] > 0:
                _, _, best_angle = self.calculate_costs(
                    curr['hand'], curr['finger'], curr['string'], curr['fret']
                )
                wrist_angles[i] = best_angle

        # Cost Calculation Loop
        for i in range(num_notes):
            curr = notes_data[i]
            row = {'PC': 0.0, 'SC': 0.0, 'MC': 0.0, 'ST': 0.0, 'DH': 0.0, 'NO': 0.0}
            
            time_penalty = 0.0
            
            if curr['note'] != 'R':
                prev_i = prev_idx_map.get(i, i)
                
                if prev_i == i:
                    time_penalty = 0.0 
                else:
                    t_val = acc_time[i] + 0.01
                    time_penalty = self.tp_base + (self.tp_ref / t_val) ** self.tp_exp
                
                if curr['fret'] == 0: row['NO'] = 1.0

                if curr['finger'] in [1,2,3,4] and curr['fret'] > 0:
                    st_raw, ph_raw, _ = self.calculate_costs(
                        curr['hand'], curr['finger'], curr['string'], curr['fret']
                    )
                    if st_raw >= 50000: st_raw = 100000.0
                    row['ST'] = st_raw
                    row['DH'] = ph_raw

                if prev_i != i and prev_i != -1:
                    prev = notes_data[prev_i]
                    row['PC'] = abs(self.fret_pos[prev['hand']] - self.fret_pos[curr['hand']])
                    row['SC'] = self.p['stringSpacing'] * abs(prev['string'] - curr['string'])
                    
                    # MC: Wrist angle change
                    row['MC'] = abs(wrist_angles[i] - wrist_angles[prev_i])
                    
                    # Barre chord SC adjustment (matching OPL logic)
                    if (curr['finger'] == 1 and prev['finger'] == 1 and 
                        curr['fret'] == prev['fret'] and abs(curr['string'] - prev['string']) >= 1):
                        row['SC'] -= (self.p['stringSpacing'] * 0.9)

            # Weight Application
            w_pc = row['PC'] * time_penalty * self.p['d_PC'] * self.p['alpha_PC']
            w_sc = row['SC'] * time_penalty * self.p['d_SC'] * self.p['alpha_SC']
            w_mc = row['MC'] * time_penalty * self.p['d_MC'] * self.p['alpha_MC']
            w_st = row['ST'] * self.p['d_ST'] * self.p['alpha_ST'] 
            w_dh = row['DH'] * self.p['d_DH'] * self.p['alpha_DH']
            w_no = row['NO'] * self.p['d_NO'] * self.p['alpha_NO']

            total['PC'] += w_pc
            total['SC'] += w_sc
            total['MC'] += w_mc
            total['ST'] += w_st
            total['DH'] += w_dh
            total['NO'] += w_no
            
            sum_weighted = w_pc + w_sc + w_mc + w_st + w_dh + w_no
            details.append({
                'Index': i+1, 'Note': curr['note'],
                'Time(ms)': acc_time.get(i, 0),
                'TimeFactor': time_penalty,
                'PC_w': w_pc, 'SC_w': w_sc, 'MC_w': w_mc, 'ST_w': w_st, 'PH_w': w_dh,
                'ST_raw': row['ST'], 'PH_raw': row['DH'],
                'Total': sum_weighted
            })

        grand_total = sum(total.values())
        return total, grand_total, pd.DataFrame(details)

if __name__ == "__main__":
    dat_file = "datas/complex_solo_mod.dat"
    sol_file = "results/complex_solo_mod_modified_model (39).txt"
    
    if os.path.exists(dat_file) and os.path.exists(sol_file):
        params = parse_dat_file(dat_file)
        model = GuitarModelReproducer(params)
        notes = parse_solution_file(sol_file)
        
        if notes:
            bd, total, df = model.calculate_total(notes)
            print("\n" + "="*60)
            print(" --- Cost Breakdown (Matching modified_model.mod) ---")
            print(f"  Position Change (PC): {bd['PC']:.4f}")
            print(f"  Finger Move     (MC): {bd['MC']:.4f}")
            print(f"  String Change   (SC): {bd['SC']:.4f}")
            print(f"  Geo Stretch     (ST): {bd['ST']:.4f}")
            print(f"  Phys Action     (DH): {bd['DH']:.4f}")
            print(f"  Open String     (NO): {bd['NO']:.4f}")
            print("-" * 40)
            print(f"  Total Cost (Sum)    : {total:.4f}")
            print("="*60)
            
            if not df.empty:
                print("\n[Details - All Notes]")
                pd.set_option('display.max_columns', None)
                pd.set_option('display.width', 1000)
                pd.set_option('display.max_rows', None)
                
                print(df[['Index', 'Note', 'Time(ms)', 'TimeFactor', 'PC_w', 'MC_w', 'SC_w', 'ST_w', 'PH_w', 'Total']].to_string(index=False, float_format="%.3f"))
    else:
        print("Please upload 'datas/complex_solo_mod.dat' and solution file to run the calculation.")
